<a href="https://colab.research.google.com/github/TuNgoc233/ETL_CaNhan_2/blob/master/Offline_recommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Offline Recommender Pipeline - 5-Star Hybrid

He thong goi y offline theo flow thong nhat:

1. Setup
2. Load & Merge Metadata
3. Content-Based
4. Collaborative Filtering - Funk SVD
5. Hybrid Union (CB, CF)
6. Test
7. Danh gia mo hinh bang RMSE cho Content-Based va Collaborative Filtering
8. Trich xuat du lieu mau ra CSV

Luu y: toan bo rating duoc chuan hoa ve thang 5 sao truoc khi train, danh gia va export sample.

## 1. Setup

In [ ]:
# === Google Colab setup ===
# 1) Mount Google Drive (chỉ chạy trên Colab)
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# 2) Cài thư viện Colab chưa có sẵn
if IN_COLAB:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'sentence-transformers'], check=True)

import os, json, pickle, time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import load_npz
from sentence_transformers import SentenceTransformer

# 3) Đường dẫn — giả định folder `Recommendation_System` nằm trực tiếp trong `MyDrive`
#    (tức là Drive sẽ có: MyDrive/Recommendation_System/places_lookup.csv, ...)
if IN_COLAB:
    BASE = Path('/content/drive/MyDrive/Recommender System')
else:
    BASE = Path('.').resolve()

POI_DIR = BASE / 'DATN-ETL' / 'Filtered_Data_Foody'
RES_DIR = BASE / 'DATN-ETL' / 'merged_foody_restaurant'
OUT_DIR = BASE / 'recommender_artifacts'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Sanity check — báo lỗi sớm nếu thiếu file
for p in [BASE / 'places_lookup.csv',
          BASE / 'rating_matrix.npz',
          BASE / 'rating_matrix_users.csv',
          BASE / 'rating_matrix_items.csv',
          POI_DIR, RES_DIR]:
    assert p.exists(), f'Không tìm thấy: {p}'

print('BASE:', BASE)
print('OUT :', OUT_DIR)

## 2. Load & Merge Metadata

In [ ]:
# 1) places_lookup
places_lookup = pd.read_csv(BASE / 'places_lookup.csv', encoding='utf-8-sig')
places_lookup['ResID'] = places_lookup['ResID'].astype(int)

# 2) Scan POI -> dict[res_id] = (category_group, category_sub)
poi_meta = {}
for city_dir in sorted(POI_DIR.iterdir()):
    if not city_dir.is_dir():
        continue
    for jf in city_dir.glob('*.json'):
        try:
            with open(jf, 'r', encoding='utf-8') as f:
                data = json.load(f)
        except Exception as e:
            print('skip', jf, e); continue
        for rec in data:
            rid = rec.get('id')
            if rid is None:
                continue
            poi_meta[int(rid)] = (rec.get('category_group') or '', rec.get('category_sub') or '')

# 3) Scan restaurants -> set
res_ids_set = set()
for jf in sorted(RES_DIR.glob('*.json')):
    try:
        with open(jf, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception as e:
        print('skip', jf, e); continue
    for rec in data:
        rid = rec.get('Id')
        if rid is None:
            continue
        res_ids_set.add(int(rid))

# 4) Merge -> places_meta
rows = []
for _, r in places_lookup.iterrows():
    rid = int(r['ResID'])
    name, city = r['PlaceName'], r['City']
    if rid in poi_meta:
        cg, cs = poi_meta[rid]
        rows.append((rid, name, city, cg, cs, 'POI'))
    elif rid in res_ids_set:
        rows.append((rid, name, city, 'Nhà hàng', '', 'Res'))
    else:
        rows.append((rid, name, city, '', '', 'Unknown'))

places_meta = pd.DataFrame(rows, columns=['ResID','PlaceName','City','CategoryGroup','CategorySub','Kind'])
places_meta = places_meta.drop_duplicates('ResID').reset_index(drop=True)
for c in ('PlaceName','City','CategoryGroup','CategorySub'):
    places_meta[c] = places_meta[c].fillna('').astype(str)

# Mapping ResID -> City (dùng cho hybrid filter online)
res2city = dict(zip(places_meta['ResID'].astype(int).values, places_meta['City'].values))

# Save metadata
places_meta.to_pickle(OUT_DIR / 'places_meta.pkl')

print(f'places_meta: {places_meta.shape}')
print(places_meta['Kind'].value_counts())
print(f'Cities: {places_meta["City"].nunique()}')
print(places_meta.head())

places_meta: (38936, 6)
Kind
Res        36519
POI         2416
Unknown        1
Name: count, dtype: int64
Cities: 65
     ResID                    PlaceName       City CategoryGroup  \
0  1057377             Đức Luyện Mobile  Hải Dương       Mua Sắm   
1   140097  Linh Nhi Spa - Bùi Thị Xuân  Hải Dương      Đẹp Khỏe   
2   140133        Solus Spa - Tam Giang  Hải Dương      Đẹp Khỏe   
3   213372     Khu Di Tích Phượng Hoàng  Hải Dương       Du Lịch   
4    23351          Hoa Phượng Karaoke   Hải Dương      Giải Trí   

        CategorySub Kind  
0               Chợ  POI  
1     Spa & Massage  POI  
2     Spa & Massage  POI  
3  Bảo tàng di tích  POI  
4           Karaoke  POI  


## 3. Content-Based — Semantic Embedding

**Quy trình**:
1. Tạo `content_text = "PlaceName - CategoryGroup - CategorySub"`.
2. Encode bằng `paraphrase-multilingual-MiniLM-L12-v2` rồi L2-normalize → cosine = dot product.
3. Hàm tra cứu `get_top_k_items_by_item(item_id, k)` → tính cosine giữa item và các địa điểm cùng thành phố, lấy top-K.
4. Pre-compute lookup table `Item_ID → List[Similar_Item_IDs]` và lưu xuống file.

In [ ]:
# 1) Content text
places_meta['content_text'] = places_meta.apply(
    lambda r: f"{r['PlaceName']} - {r['CategoryGroup']} - {r['CategorySub']}".strip(),
    axis=1,
)

# 2) Encode embeddings (Transformer + L2-normalize)
EMBED_MODEL_NAME = 'paraphrase-multilingual-MiniLM-L12-v2'
print(f'Loading {EMBED_MODEL_NAME}...')
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

print('Encoding embeddings...')
embeddings = embed_model.encode(
    places_meta['content_text'].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,                          # L2-norm -> cosine = dot
).astype(np.float32)
print(f'embeddings: {embeddings.shape}, dtype={embeddings.dtype}')

# Map ResID -> embedding row index
res2content_idx = {int(rid): i for i, rid in enumerate(places_meta['ResID'].values)}
city_arr        = places_meta['City'].values            # vectorised city lookup

# 3) Hàm tra cứu — Cosine similarity, lọc cùng thành phố
def get_top_k_items_by_item(item_id: int, k: int = 100) -> list:
    """Trả về top-K ResID có cosine cao nhất với item_id, trong CÙNG thành phố.

    Quy trình:
      - Lấy vector embedding của item_id.
      - Lọc tập ứng viên về các địa điểm cùng city.
      - Tính cosine = dot product giữa vector item và ma trận embeddings ứng viên.
      - Sắp xếp giảm dần, loại chính item, cắt top-K.
    """
    item_id = int(item_id)
    idx = res2content_idx.get(item_id)
    if idx is None:
        return []
    city = res2city.get(item_id)
    if not city:
        return []
    cand_idx = np.where(city_arr == city)[0]
    if len(cand_idx) == 0:
        return []
    sim = embeddings[idx] @ embeddings[cand_idx].T      # cosine vì đã L2-norm
    order = np.argsort(-sim)
    cand_resids = places_meta['ResID'].values[cand_idx][order]
    return cand_resids[cand_resids != item_id][:k].astype(int).tolist()

# 4) Pre-compute lookup table: Item_ID -> List[Similar_Item_IDs]
TOP_K_CB = 50
print(f'\nPre-computing CB lookup (k={TOP_K_CB})...')
cb_lookup = {}
all_resids = places_meta['ResID'].astype(int).values
t0 = time.time()
for i, rid in enumerate(all_resids, 1):
    cb_lookup[int(rid)] = get_top_k_items_by_item(int(rid), k=TOP_K_CB)
    if i % 5000 == 0 or i == len(all_resids):
        print(f'  [{i}/{len(all_resids)}] elapsed={time.time()-t0:.1f}s', flush=True)

# 5) Save artifacts
np.save(OUT_DIR / 'content_embeddings.npy', embeddings)
with open(OUT_DIR / 'cb_lookup.pkl', 'wb') as f:
    pickle.dump(cb_lookup, f)
with open(OUT_DIR / 'embedding_model.txt', 'w', encoding='utf-8') as f:
    f.write(EMBED_MODEL_NAME)
print(f'\nCB lookup: {len(cb_lookup)} items, top-{TOP_K_CB} each → cb_lookup.pkl')

Loading paraphrase-multilingual-MiniLM-L12-v2...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding embeddings...


Batches:   0%|          | 0/1217 [00:00<?, ?it/s]

embeddings: (38936, 384), dtype=float32

Pre-computing CB lookup (k=50)...
  [5000/38936] elapsed=10.6s
  [10000/38936] elapsed=20.5s
  [15000/38936] elapsed=29.2s
  [20000/38936] elapsed=37.0s
  [25000/38936] elapsed=44.7s
  [30000/38936] elapsed=52.8s
  [35000/38936] elapsed=60.8s
  [38936/38936] elapsed=66.8s

CB lookup: 38936 items, top-50 each → cb_lookup.pkl


## 4. Collaborative Filtering - Funk SVD (Surprise)

Phan nay load ma tran rating, chuan hoa rating ve thang 5 sao, split train/validation/test, train Funk SVD va pre-compute lookup CF.

In [ ]:
!pip install numpy==1.26.4 Cython -q
!pip uninstall -y scikit-surprise
!pip install scikit-surprise --no-binary scikit-surprise --no-build-isolation -q

import math
import random
import time
import pickle

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, load_npz
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error

R_raw = load_npz(BASE / 'rating_matrix.npz').astype(np.float32)
R = R_raw.copy().astype(np.float32)

# Chuan hoa rating ve thang 5 sao.
# Neu du lieu goc dang o thang 10 thi chia 2; neu da la 5 sao thi giu nguyen.
rating_max = float(R.data.max()) if R.nnz else 0.0
if rating_max > 5.0:
    R.data = R.data / 2.0
R.data = np.clip(R.data, 0.5, 5.0).astype(np.float32)

cf_users = pd.read_csv(BASE / 'rating_matrix_users.csv', encoding='utf-8-sig')
cf_items = pd.read_csv(BASE / 'rating_matrix_items.csv', encoding='utf-8-sig')
cf_users.columns = [c.strip() for c in cf_users.columns]
cf_items.columns = [c.strip() for c in cf_items.columns]
cf_user_ids = cf_users.iloc[:, 0].astype(int).values
cf_item_ids = cf_items.iloc[:, 0].astype(int).values
R_csr = R.tocsr()

print(f'R: {R.shape}, nnz={R.nnz}, users={len(cf_user_ids)}, items={len(cf_item_ids)}')
print(f'Rating scale after normalization: min={R.data.min():.2f}, max={R.data.max():.2f}')

R_coo = R_csr.tocoo()
df_ratings = pd.DataFrame({
    'u_idx': R_coo.row,
    'i_idx': R_coo.col,
    'rating': R_coo.data,
})

reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(df_ratings[['u_idx', 'i_idx', 'rating']], reader)

raw_ratings = list(data.raw_ratings)
random.seed(42)
random.shuffle(raw_ratings)

total = len(raw_ratings)
test_size = int(0.1 * total)
val_size = int(0.1 * total)
train_size = total - test_size - val_size

train_raw = raw_ratings[:train_size]
val_raw = raw_ratings[train_size:train_size + val_size]
test_raw = raw_ratings[train_size + val_size:]

data.raw_ratings = train_raw
train_set = data.build_full_trainset()
val_set = data.construct_testset(val_raw)
test_set = data.construct_testset(test_raw)

print(f'\nTong so ratings: {total}')
print(f'Train set size: {len(train_raw)}')
print(f'Validation set size: {len(val_raw)}')
print(f'Test set size: {len(test_raw)}')

print('\nDang xay dung R_train_csr tu train set de tranh data leakage...')
train_u_indices = [uid for uid, _, _, _ in train_raw]
train_i_indices = [iid for _, iid, _, _ in train_raw]
train_ratings_data = [rating for _, _, rating, _ in train_raw]

num_users, num_items = R_csr.shape
R_train_csr = csr_matrix(
    (train_ratings_data, (train_u_indices, train_i_indices)),
    shape=(num_users, num_items),
    dtype=np.float32,
)
global_mean_5star = float(np.mean(train_ratings_data))

print('\nDang tim kiem sieu tham so toi uu cho Funk SVD tren train set...')
param_grid = {
    'n_factors': [16, 32],
    'n_epochs': [10, 20],
    'lr_all': [0.002, 0.005],
    'reg_all': [0.05, 0.1, 0.2],
}

gs = GridSearchCV(SVD, param_grid, measures=['rmse'], cv=3, n_jobs=-1)
gs.fit(data)

best_svd_params = gs.best_params['rmse']
print('Bo sieu tham so tot nhat:', best_svd_params)

cf_algo = SVD(**best_svd_params)
cf_algo.fit(train_set)

print('Dang dong bo User/Item Factors sang ma tran NumPy de nhan nhanh...')
P = np.zeros((num_users, cf_algo.pu.shape[1]), dtype=np.float32)
b_u = np.zeros(num_users, dtype=np.float32)
for raw_u in range(num_users):
    try:
        inner_u = train_set.to_inner_uid(raw_u)
        P[raw_u] = cf_algo.pu[inner_u]
        b_u[raw_u] = cf_algo.bu[inner_u]
    except ValueError:
        pass

Q = np.zeros((num_items, cf_algo.qi.shape[1]), dtype=np.float32)
b_i = np.zeros(num_items, dtype=np.float32)
for raw_i in range(num_items):
    try:
        inner_i = train_set.to_inner_iid(raw_i)
        Q[raw_i] = cf_algo.qi[inner_i]
        b_i[raw_i] = cf_algo.bi[inner_i]
    except ValueError:
        pass

global_mean = train_set.global_mean
user_id_to_cf_idx = {int(uid): i for i, uid in enumerate(cf_user_ids)}

def get_top_k_items_for_user_svd(user_id: int, k: int = 100) -> list:
    uid = int(user_id)
    u_idx = user_id_to_cf_idx.get(uid)
    if u_idx is None:
        return []

    preds = global_mean + b_u[u_idx] + b_i + Q.dot(P[u_idx])

    start, end = R_csr.indptr[u_idx], R_csr.indptr[u_idx + 1]
    history_items = R_csr.indices[start:end]
    preds[history_items] = -np.inf

    if k >= len(preds):
        order = np.argsort(-preds)
    else:
        part = np.argpartition(-preds, kth=k)[:k]
        order = part[np.argsort(-preds[part])]

    order = order[preds[order] > -np.inf]
    return cf_item_ids[order].astype(int).tolist()

TOP_K_CF = 300
cf_lookup = {}
t0 = time.time()
print(f'\nPre-computing CF lookup (k={TOP_K_CF})...')
for i, uid in enumerate(cf_user_ids, 1):
    cf_lookup[int(uid)] = get_top_k_items_for_user_svd(int(uid), k=TOP_K_CF)
    if i % 10000 == 0 or i == len(cf_user_ids):
        print(f'  [{i}/{len(cf_user_ids)}] elapsed={time.time()-t0:.1f}s', flush=True)

np.save(OUT_DIR / 'cf_user_factors.npy', P)
np.save(OUT_DIR / 'cf_item_factors.npy', Q)
pd.DataFrame({'UserID': cf_user_ids}).to_csv(OUT_DIR / 'cf_user_ids.csv', index=False, encoding='utf-8-sig')
pd.DataFrame({'ResID':  cf_item_ids}).to_csv(OUT_DIR / 'cf_item_ids.csv', index=False, encoding='utf-8-sig')
with open(OUT_DIR / 'cf_lookup.pkl', 'wb') as f:
    pickle.dump(cf_lookup, f)

print(f'\nCF lookup: {len(cf_lookup)} users, top-{TOP_K_CF} each -> cf_lookup.pkl')

## 5. Hybrid — Union(CB, CF)

**Đầu vào**: `(user_id, current_item_id)`.

**Quy trình**:
1. Tra cứu song song:
   - `cb_list = cb_lookup[current_item_id]`  (đã trong cùng city)
   - `cf_list = cf_lookup[user_id]`          (toàn cục)
2. Filter `cf_list` về cùng thành phố với `current_item_id`.
3. **Union** với priority: items xuất hiện trong **CẢ HAI** list → CB-only → CF-only, dedupe, cắt top-K.

**Đầu ra**: DataFrame xếp hạng kèm cột `Source ∈ {BOTH, CB, CF}`.

In [ ]:
def recommend_hybrid(user_id: int, current_item_id: int, k: int = 10) -> pd.DataFrame:
    """Hybrid: Union(CB, CF) cùng thành phố, ưu tiên giao điểm.

    Tra cứu song song 2 lookup table -> Union với priority:
      1. Items có trong CẢ CB và CF (BOTH)
      2. CB-only
      3. CF-only (đã filter cùng city)
    """
    user_id         = int(user_id)
    current_item_id = int(current_item_id)

    target_city = res2city.get(current_item_id)
    if target_city is None:
        return pd.DataFrame()

    # Truy xuất song song
    cb_list = [int(r) for r in cb_lookup.get(current_item_id, []) if int(r) != current_item_id]
    cf_full = [int(r) for r in cf_lookup.get(user_id,         []) if int(r) != current_item_id]
    # Filter CF về cùng city
    cf_list = [r for r in cf_full if res2city.get(r) == target_city]

    cb_set, cf_set = set(cb_list), set(cf_list)
    intersect = cb_set & cf_set

    # Union, priority: BOTH -> CB-only -> CF-only
    ordered, seen = [], set()
    for r in cb_list:                                       # giao điểm trước (theo thứ tự CB)
        if r in intersect and r not in seen:
            ordered.append(r); seen.add(r)
    for r in cb_list:                                       # CB-only
        if r not in seen:
            ordered.append(r); seen.add(r)
    for r in cf_list:                                       # CF-only (đã cùng city)
        if r not in seen:
            ordered.append(r); seen.add(r)

    final = ordered[:k]
    rows = []
    for rank, rid in enumerate(final, 1):
        tag = 'BOTH' if rid in intersect else ('CB' if rid in cb_set else 'CF')
        rows.append({'ResID': rid, 'Rank': rank, 'Source': tag})

    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df = df.merge(
        places_meta[['ResID','PlaceName','City','CategoryGroup','CategorySub']],
        on='ResID', how='left',
    )
    return df[['ResID','PlaceName','City','CategoryGroup','CategorySub','Rank','Source']]

print('Defined recommend_hybrid()')

Defined recommend_hybrid()


## 6. Test

Chọn vài user có nhiều ratings, dùng item đầu tiên họ đã rate làm `current_item_id`, gọi `recommend_hybrid()` và in ra kết quả.

In [ ]:
user_counts = np.diff(R_csr.indptr)
top_users = np.argsort(-user_counts)[:5]

for u_idx in top_users[:3]:
    uid = int(cf_user_ids[u_idx])
    rated = R_csr[u_idx].indices
    if len(rated) == 0:
        continue
    current_item = int(cf_item_ids[rated[0]])
    src = places_meta[places_meta['ResID'] == current_item]
    if src.empty:
        continue
    src_row = src.iloc[0]
    print(f"\n=== user={uid} | current_item={current_item} "
          f"({src_row['PlaceName']} — {src_row['City']}) ===")
    df = recommend_hybrid(uid, current_item, k=10)
    if df.empty:
        print('(no recommendations)')
    else:
        print(df[['PlaceName','City','Rank','Source']].to_string(index=False))


=== user=873230 | current_item=235 (Thùy Vân Hotel & Restaurant — Vũng Tàu) ===
                          PlaceName     City  Rank Source
Nhà Hàng RIVA - Vũng Tàu RIVA Hotel Vũng Tàu     1     CB
              Quán Kiều - Bò lá Lốt Vũng Tàu     2     CB
    Hải Phòng Quán - Lương Thế Vinh Vũng Tàu     3     CB
                    Quán Nhậu Bờ Kè Vũng Tàu     4     CB
                       Phở Minh Tâm Vũng Tàu     5     CB
                 Hải Anh Restaurant Vũng Tàu     6     CB
              Mai Tồ - Phở & Lẩu Bò Vũng Tàu     7     CB
   Tịnh Tâm Quán - Cafe Nguyên Chất Vũng Tàu     8     CB
                             Phở Kỳ Vũng Tàu     9     CB
                  Quán Bún Xuân Thu Vũng Tàu    10     CB

=== user=859698 | current_item=4728 (Ô Cấp 1 Cafe — Vũng Tàu) ===
         PlaceName     City  Rank Source
Hương Tràm 1 Cafe  Vũng Tàu     1     CB
        Ô Cấp Cafe Vũng Tàu     2     CB
          Xưa Cafe Vũng Tàu     3     CB
   Cát Tường Cafe  Vũng Tàu     4     CB
         

## 7. Danh gia mo hinh - RMSE

Danh gia rating prediction tren thang 5 sao:

- Collaborative Filtering: dung Funk SVD tu thu vien Surprise.
- Content-Based: mo phong du doan rating bang item-based kNN tren embedding, chi dung `R_train_csr` de tranh data leakage.
- Hybrid RMSE duoc tinh them de tham khao, con Hybrid Union o buoc 5 van la flow goi y Top-K chinh.

In [ ]:
def predict_content_based_rating(uid, iid, R_matrix, global_mean_rating):
    start, end = R_matrix.indptr[int(uid)], R_matrix.indptr[int(uid) + 1]
    user_rated_items = R_matrix.indices[start:end]
    user_ratings = R_matrix.data[start:end]

    target_resid = int(cf_item_ids[int(iid)])
    target_emb_idx = res2content_idx.get(target_resid)
    if target_emb_idx is None or len(user_rated_items) == 0:
        return float(global_mean_rating)

    rated_emb_indices = []
    rated_values = []
    for item_idx, rating_value in zip(user_rated_items, user_ratings):
        rated_resid = int(cf_item_ids[int(item_idx)])
        emb_idx = res2content_idx.get(rated_resid)
        if emb_idx is not None:
            rated_emb_indices.append(emb_idx)
            rated_values.append(float(rating_value))

    if not rated_emb_indices:
        return float(global_mean_rating)

    sims = embeddings[rated_emb_indices] @ embeddings[target_emb_idx]
    positive_mask = sims > 0
    if not np.any(positive_mask):
        pred = np.mean(rated_values)
    else:
        sims_pos = sims[positive_mask]
        ratings_pos = np.array(rated_values, dtype=np.float32)[positive_mask]
        pred = float(np.dot(sims_pos, ratings_pos) / np.sum(sims_pos))

    return float(np.clip(pred, 0.5, 5.0))

print('Dang danh gia CF (Funk SVD)...')
train_predictions_cf = cf_algo.test(train_set.build_testset())
val_predictions_cf = cf_algo.test(val_set)
test_predictions_cf = cf_algo.test(test_set)

cf_train_rmse = accuracy.rmse(train_predictions_cf, verbose=False)
cf_val_rmse = accuracy.rmse(val_predictions_cf, verbose=False)
cf_test_rmse = accuracy.rmse(test_predictions_cf, verbose=False)

print('Dang danh gia Content-Based kNN...')
true_ratings_val = [true_r for uid, iid, true_r in val_set]
cb_preds_val = [predict_content_based_rating(uid, iid, R_train_csr, global_mean_5star) for uid, iid, _ in val_set]
cb_val_rmse = math.sqrt(mean_squared_error(true_ratings_val, cb_preds_val))

true_ratings_test = [true_r for uid, iid, true_r in test_set]
cb_preds_test = [predict_content_based_rating(uid, iid, R_train_csr, global_mean_5star) for uid, iid, _ in test_set]
cb_test_rmse = math.sqrt(mean_squared_error(true_ratings_test, cb_preds_test))

print('Dang toi uu alpha Hybrid RMSE tren validation set...')
cf_preds_val = [pred.est for pred in val_predictions_cf]
cf_preds_test = [pred.est for pred in test_predictions_cf]

alphas = np.arange(0.0, 1.01, 0.05)
best_alpha = 0.0
best_hybrid_val_rmse = float('inf')
for alpha in alphas:
    hybrid_val_preds = [alpha * cf + (1 - alpha) * cb for cf, cb in zip(cf_preds_val, cb_preds_val)]
    hybrid_val_rmse = math.sqrt(mean_squared_error(true_ratings_val, hybrid_val_preds))
    if hybrid_val_rmse < best_hybrid_val_rmse:
        best_hybrid_val_rmse = hybrid_val_rmse
        best_alpha = float(alpha)

hybrid_test_preds = [best_alpha * cf + (1 - best_alpha) * cb for cf, cb in zip(cf_preds_test, cb_preds_test)]
hybrid_test_rmse = math.sqrt(mean_squared_error(true_ratings_test, hybrid_test_preds))

rmse_df = pd.DataFrame([
    {'Model': 'Collaborative Filtering (Funk SVD)', 'Split': 'Train', 'RMSE': cf_train_rmse},
    {'Model': 'Collaborative Filtering (Funk SVD)', 'Split': 'Validation', 'RMSE': cf_val_rmse},
    {'Model': 'Collaborative Filtering (Funk SVD)', 'Split': 'Test', 'RMSE': cf_test_rmse},
    {'Model': 'Content-Based kNN', 'Split': 'Validation', 'RMSE': cb_val_rmse},
    {'Model': 'Content-Based kNN', 'Split': 'Test', 'RMSE': cb_test_rmse},
    {'Model': f'Hybrid Weighted (alpha={best_alpha:.2f})', 'Split': 'Validation', 'RMSE': best_hybrid_val_rmse},
    {'Model': f'Hybrid Weighted (alpha={best_alpha:.2f})', 'Split': 'Test', 'RMSE': hybrid_test_rmse},
])

rmse_df.to_csv(OUT_DIR / 'rmse_results_5star.csv', index=False, encoding='utf-8-sig')
print('\n=== KET QUA RMSE THANG 5 SAO ===')
print(rmse_df.to_string(index=False))
print('\nBest SVD Params:', best_svd_params)
print(f'Saved: {OUT_DIR / "rmse_results_5star.csv"}')

### 7.1 Inspect predicted vs actual ratings and leakage/overfitting

Cell nay in mau rating thuc te va rating du doan cho mot user cu the, sau do dua ra chan doan nhanh ve overfitting va data leakage.

In [ ]:
from collections import Counter

USER_ID_TO_INSPECT = None  # Doi thanh UserID cu the, vi du: 12345. De None se tu chon user co nhieu rating trong test set.
N_PREDICTION_EXAMPLES = 10


def pick_user_from_split(split_set, user_id=None):
    if user_id is not None:
        target = int(user_id)
        matched = [(uid, iid, true_r) for uid, iid, true_r in split_set if int(cf_user_ids[int(uid)]) == target]
        if matched:
            return target, matched
        print(f'Khong tim thay UserID={target} trong split dang inspect; tu dong chon user khac.')

    counts = Counter(int(uid) for uid, _, _ in split_set)
    selected_uid_idx, _ = counts.most_common(1)[0]
    selected_user_id = int(cf_user_ids[selected_uid_idx])
    matched = [(uid, iid, true_r) for uid, iid, true_r in split_set if int(uid) == selected_uid_idx]
    return selected_user_id, matched


def build_prediction_inspection(split_set, user_id=None, n=10):
    selected_user_id, samples = pick_user_from_split(split_set, user_id=user_id)
    rows = []
    for uid, iid, true_r in samples[:n]:
        uid_idx = int(uid)
        iid_idx = int(iid)
        resid = int(cf_item_ids[iid_idx])
        meta = places_meta.loc[places_meta['ResID'].astype(int) == resid]
        if meta.empty:
            place_name, city = '', ''
        else:
            place_name = meta.iloc[0]['PlaceName']
            city = meta.iloc[0]['City']

        cf_pred = float(np.clip(cf_algo.predict(uid, iid).est, 0.5, 5.0))
        cb_pred = float(predict_content_based_rating(uid_idx, iid_idx, R_train_csr, global_mean_5star))
        hybrid_pred = float(np.clip(best_alpha * cf_pred + (1 - best_alpha) * cb_pred, 0.5, 5.0))

        rows.append({
            'UserID': int(cf_user_ids[uid_idx]),
            'ResID': resid,
            'PlaceName': place_name,
            'City': city,
            'Actual_5Star': round(float(true_r), 3),
            'CF_Pred_5Star': round(cf_pred, 3),
            'CB_Pred_5Star': round(cb_pred, 3),
            'Hybrid_Pred_5Star': round(hybrid_pred, 3),
            'CF_Abs_Error': round(abs(cf_pred - float(true_r)), 3),
            'CB_Abs_Error': round(abs(cb_pred - float(true_r)), 3),
            'Hybrid_Abs_Error': round(abs(hybrid_pred - float(true_r)), 3),
        })

    return selected_user_id, pd.DataFrame(rows)


inspect_user_id, prediction_examples_df = build_prediction_inspection(
    test_set,
    user_id=USER_ID_TO_INSPECT,
    n=N_PREDICTION_EXAMPLES,
)

prediction_examples_df.to_csv(OUT_DIR / 'prediction_examples_5star.csv', index=False, encoding='utf-8-sig')
print(f'Prediction examples for UserID={inspect_user_id} on TEST split')
display(prediction_examples_df)

cf_train_val_gap = cf_val_rmse - cf_train_rmse
cf_train_test_gap = cf_test_rmse - cf_train_rmse
cf_val_test_gap = abs(cf_test_rmse - cf_val_rmse)
cb_val_test_gap = abs(cb_test_rmse - cb_val_rmse)
hybrid_val_test_gap = abs(hybrid_test_rmse - best_hybrid_val_rmse)

if cf_train_test_gap > 0.30:
    overfit_status = 'Canh bao: CF co dau hieu overfitting ro, train RMSE thap hon test kha nhieu.'
elif cf_train_test_gap > 0.15:
    overfit_status = 'Theo doi: CF co khoang cach train-test vua phai, can kiem tra them theo tung nhom user/item.'
else:
    overfit_status = 'On: chua thay dau hieu overfitting manh qua RMSE train/validation/test.'

if max(cf_val_test_gap, cb_val_test_gap, hybrid_val_test_gap) > 0.15:
    generalization_status = 'Validation va test lech nhau tuong doi lon; nen kiem tra lai cach split hoac do phan bo data.'
else:
    generalization_status = 'Validation va test kha gan nhau; kha nang tong quat hoa on dinh hon.'

leakage_notes = [
    'CF: train bang train_set, RMSE tinh tren validation/test rieng nen khong thay leakage truc tiep trong rating prediction.',
    'CB: predict rating chi dung R_train_csr lam lich su user, tranh dung rating validation/test lam dau vao.',
    'Metadata/content embeddings dung thong tin item, khong dung rating muc tieu; day khong phai rating leakage.',
    'Luu y: recommend_hybrid/CF lookup dang mask item da tuong tac bang R_csr day du. Neu dung de danh gia ranking holdout, nen mask bang R_train_csr de tranh nhin thay lich su validation/test.',
]

diagnosis_df = pd.DataFrame([
    {'Check': 'CF train RMSE', 'Value': round(cf_train_rmse, 4), 'Interpretation': 'Baseline fit tren train'},
    {'Check': 'CF validation RMSE', 'Value': round(cf_val_rmse, 4), 'Interpretation': 'Sai so tren validation'},
    {'Check': 'CF test RMSE', 'Value': round(cf_test_rmse, 4), 'Interpretation': 'Sai so tren test'},
    {'Check': 'CF train-test gap', 'Value': round(cf_train_test_gap, 4), 'Interpretation': overfit_status},
    {'Check': 'CF val-test gap', 'Value': round(cf_val_test_gap, 4), 'Interpretation': generalization_status},
    {'Check': 'CB val-test gap', 'Value': round(cb_val_test_gap, 4), 'Interpretation': generalization_status},
    {'Check': 'Hybrid val-test gap', 'Value': round(hybrid_val_test_gap, 4), 'Interpretation': generalization_status},
])

diagnosis_df.to_csv(OUT_DIR / 'overfitting_leakage_diagnosis.csv', index=False, encoding='utf-8-sig')
print('\n=== OVERFITTING / DATA LEAKAGE DIAGNOSIS ===')
display(diagnosis_df)
print('\nLeakage notes:')
for note in leakage_notes:
    print('-', note)
print(f'\nSaved: {OUT_DIR / "prediction_examples_5star.csv"}')
print(f'Saved: {OUT_DIR / "overfitting_leakage_diagnosis.csv"}')

## 8. Trich xuat du lieu mau ra CSV

In [ ]:
SAMPLE_DIR = OUT_DIR / 'samples_csv'
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

print('Dang trich xuat du lieu mau ra CSV...')

places_meta.head(100).to_csv(SAMPLE_DIR / 'sample_places_meta.csv', index=False, encoding='utf-8-sig')

R_coo = R.tocoo()
cf_input_df = pd.DataFrame({
    'UserID': cf_user_ids[R_coo.row[:100]],
    'ResID': cf_item_ids[R_coo.col[:100]],
    'Rating_5Star': R_coo.data[:100],
})
cf_input_df.to_csv(SAMPLE_DIR / 'sample_cf_input_ratings_5star.csv', index=False, encoding='utf-8-sig')

cb_sample = {k: cb_lookup[k] for k in list(cb_lookup.keys())[:100]}
cb_df = pd.DataFrame([{'ResID': k, 'Similar_Items': v} for k, v in cb_sample.items()])
cb_df.to_csv(SAMPLE_DIR / 'sample_cb_lookup.csv', index=False, encoding='utf-8-sig')

cf_sample = {k: cf_lookup.get(k, []) for k in list(cf_lookup.keys())[:100]}
cf_df = pd.DataFrame([{'UserID': k, 'Recommended_Items': v} for k, v in cf_sample.items()])
cf_df.to_csv(SAMPLE_DIR / 'sample_cf_lookup.csv', index=False, encoding='utf-8-sig')

pd.DataFrame(P[:100]).to_csv(SAMPLE_DIR / 'sample_user_factors_P.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(Q[:100]).to_csv(SAMPLE_DIR / 'sample_item_factors_Q.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(embeddings[:100]).to_csv(SAMPLE_DIR / 'sample_content_embeddings.csv', index=False, encoding='utf-8-sig')

if 'rmse_df' in globals():
    rmse_df.to_csv(SAMPLE_DIR / 'sample_rmse_results_5star.csv', index=False, encoding='utf-8-sig')
if 'prediction_examples_df' in globals():
    prediction_examples_df.to_csv(SAMPLE_DIR / 'sample_prediction_examples_5star.csv', index=False, encoding='utf-8-sig')
if 'diagnosis_df' in globals():
    diagnosis_df.to_csv(SAMPLE_DIR / 'sample_overfitting_leakage_diagnosis.csv', index=False, encoding='utf-8-sig')

print(f'Da luu thanh cong cac file CSV mau tai: {SAMPLE_DIR}')
display(cf_input_df.head())